# DS 3000 Project Phase I: Proposal and Data Sources

Due: Sunday Oct 4 @ 11:59 PM EST

Name: Ishita Kushwaha

1. (25%) Describes and motivates a real-world problem where data science may provide helpful insights. This problem must consist of at least two key questions of interest and your description of the problem and questions should be easily understood by a casual reader. Citations to motivating sources are preferred where possible (e.g. news articles, published papers, etc. Do not use Wikipedia itself, but the links Wikipedia articles cite may be useful)

# Clinical Trial Outcomes: What Predicts Enrollment Size and Early Termination?

## 1. Problem Description and Motivation

Before a new drug or treatment reaches patients, it has to be tested in clinical trials, which are expensive, slow, and depend on enough volunteers signing up. Many trials never finish. More than a quarter of randomized controlled trials are stopped early, mostly because too few patients are recruited [1]. In one study of cardiovascular trials listed on ClinicalTrials.gov, 684 of 6,279 trials (10.9%) were terminated early, and low recruitment was the main reason [3]. A separate analysis of the same registry found that poor accrual was the most common reason trials failed to complete (41%) [4].

This matters because a trial that stops early wastes funding, the time of researchers, and the contributions of the patients who did enroll, while the treatment question stays unanswered. Researchers have also found that these failures are often avoidable: the most common reasons for recruitment failure were overestimating how many eligible participants exist and negative views of the trial's interventions among recruiters and patients, and most reasons seemed preventable with a pilot study [2]. Other work points to design factors, with narrow eligibility criteria, investigator sponsorship, and reduced motivation of patients and recruiters linked to poor recruitment [1]. In one real example, a sickle cell disease trial was ended early due to low enrollment, with barriers including insufficient staff and competing protocols [5].

Most of this research looks at specific disease areas such as cardiovascular trials. In this project I will use data from the ClinicalTrials.gov registry on **cancer trials** to ask whether basic, easy-to-observe trial characteristics can tell us something about how these trials turn out.

## 2. Questions of Interest

1. **How large will a trial's enrollment be?** Can a trial's phase, sponsor type, number of study sites, and duration predict how many participants it enrolls?
2. **Will a trial finish or end early?** Can a trial's phase, sponsor type, study type, enrollment size, and number of sites predict whether it is Completed or Terminated?

## References

1. *Comparison of randomized controlled trials discontinued or revised for poor recruitment and completed trials with the same research question: a matched qualitative study.* [PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC6937940)
2. *A systematic review of discontinued trials suggested that most reasons for recruitment failure were preventable.* [PubMed](https://pubmed.ncbi.nlm.nih.gov/27498376/)
3. *Prevalence, characteristics, and predictors of early termination of cardiovascular clinical trials due to low recruitment: insights from the ClinicalTrials.gov registry.* [Duke Scholars](https://scholars.duke.edu/publication/1033788)
4. *Early termination of cardiovascular trials as a consequence of poor accrual: analysis of ClinicalTrials.gov 2006-2015.* BMJ Open. [Link](https://bmjopen.bmj.com/content/7/6/e013482)
5. *Clinical trial implementation and recruitment: Lessons learned from the early closure of a randomized clinical trial.* [Emory Library](https://open.library.emory.edu/concern/publications/bc25da96-1038-4752-a3f1-d23f6e281685)

2. (25%) Explicity state source and load data (does not have to be clean) from data source:
* Data must include at least **2 numeric** (e.g. number of friends, height, gpa, temperature, etc.) and **1 categorical** (e.g. color, class, car type, etc.) features and have at least **30 unique observations** (in other words, the final data set must be at least 30 rows and have at least 3 columns)
* Data **MUST BE COLLECTED VIA PYTHON, EITHER WITH AN API (WHICH RETURNS JSON/DICTIONARY FORMATTED DATA) OR WEB SCRAPING (VIA BEAUTIFUL SOUP)**. You may **NOT** simply download a .csv file or use `pandas.read_html()`.
* An exception might be made if there are multiple files/tables from disparate sources which will be non-trivial to clean/combine, but you *must* discuss this with the professor.
* Here are Links to an external site.a fewLinks to an external site. resources Links to an external site.for free APIs.
* Once you have successfully collected, curated, cleaned the data, you may save it as a .csv or .json in your working directory and call that final data set when implementing the ML (i.e. you do not have to call the API/do the web scraping every time you want to use the data).

## 3. Data Source

**Source:** ClinicalTrials.gov, the U.S. National Library of Medicine's registry of clinical studies.
**Access method:** the ClinicalTrials.gov API v2 (https://clinicaltrials.gov/data-api/api), called from Python using the `requests` library. The API returns JSON, which I convert into a pandas DataFrame.
**What I collected:** 600 completed or terminated cancer trials, with these features:
- Numeric: `enrollment` (number of participants), `n_sites` (number of study locations), `duration_days` (length of the trial)
- Categorical: `phase`, `sponsor_class`, `study_type`, and `status` (Completed or Terminated)

I saved the collected data as `clinical_trials.csv` so I don't need to call the API every time.

In [5]:
import requests
import pandas as pd

# --- 1. Collect data from the ClinicalTrials.gov API v2 ---
BASE = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "filter.overallStatus": "COMPLETED|TERMINATED",
    "query.cond": "cancer",
    "pageSize": 200,
    "format": "json",
}

studies, token = [], None
while len(studies) < 600:
    if token:
        params["pageToken"] = token
    r = requests.get(BASE, params=params)
    r.raise_for_status()
    data = r.json()
    studies.extend(data["studies"])
    token = data.get("nextPageToken")
    if not token:
        break

# --- 2. Pull out the fields ---
rows = []
for s in studies:
    p = s["protocolSection"]
    rows.append({
        "nct_id": p["identificationModule"]["nctId"],
        "status": p["statusModule"]["overallStatus"],
        "start": p["statusModule"].get("startDateStruct", {}).get("date"),
        "end": p["statusModule"].get("completionDateStruct", {}).get("date"),
        "phase": ", ".join(p.get("designModule", {}).get("phases", [])),
        "study_type": p.get("designModule", {}).get("studyType"),
        "enrollment": p.get("designModule", {}).get("enrollmentInfo", {}).get("count"),
        "sponsor_class": p.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {}).get("class"),
        "n_sites": len(p.get("contactsLocationsModule", {}).get("locations", [])),
    })

df = pd.DataFrame(rows)

# --- 3. Clean: label unphased trials so they survive the CSV round trip ---
df["phase"] = df["phase"].replace({"": "NOT_APPLICABLE", "NA": "NOT_APPLICABLE"})

# --- 4. Parse dates (handles both "2019-03" and "2019-03-15") ---
df["start"] = pd.to_datetime(df["start"], format="ISO8601", errors="coerce")
df["end"] = pd.to_datetime(df["end"], format="ISO8601", errors="coerce")
df["duration_days"] = (df["end"] - df["start"]).dt.days

# --- 5. Save ---
df.to_csv("clinical_trials.csv", index=False)
print("Saved clinical_trials.csv with shape", df.shape)

# --- 6. Quick check ---
print("\nMissing values:")
print(df.isna().sum())
print("\nStatus counts:")
print(df["status"].value_counts())
print("\nPhase counts:")
print(df["phase"].value_counts())

df.head()

Saved clinical_trials.csv with shape (600, 10)

Missing values:
nct_id            0
status            0
start             2
end              31
phase             0
study_type        0
enrollment       23
sponsor_class     0
n_sites           0
duration_days    31
dtype: int64

Status counts:
status
COMPLETED     511
TERMINATED     89
Name: count, dtype: int64

Phase counts:
phase
NOT_APPLICABLE    244
PHASE2            161
PHASE1             85
PHASE3             48
PHASE1, PHASE2     42
PHASE4             10
PHASE2, PHASE3      7
EARLY_PHASE1        3
Name: count, dtype: int64


,nct_id,status,start,end,phase,study_type,enrollment,sponsor_class,n_sites,duration_days
0,NCT00911820,COMPLETED,2009-07-01,2013-08-01,PHASE2,INTERVENTIONAL,88.0,OTHER,4,1492.0
1,NCT00343382,COMPLETED,2006-12-01,2012-12-01,PHASE3,INTERVENTIONAL,201.0,OTHER,192,2192.0
2,NCT04409782,COMPLETED,2020-10-31,2023-11-03,NOT_APPLICABLE,INTERVENTIONAL,260.0,OTHER,1,1098.0
3,NCT02403024,COMPLETED,2015-08-01,2017-09-22,NOT_APPLICABLE,INTERVENTIONAL,39.0,OTHER,2,783.0
4,NCT01408459,COMPLETED,2007-01-04,2008-09-30,NOT_APPLICABLE,INTERVENTIONAL,37.0,OTHER,1,635.0


In [6]:
df = pd.read_csv("clinical_trials.csv")

3. (25%) Write a paragraph about how the data will be used to solve the problem and your two questions of interest. At this point of the semester, we haven't studied the machine learning methods yet, but you should have a general idea of what you can do with ML (predict numerical values, predict class labels, or characterize relationships between features). If you do not, ask a TA or the professor or do a little googling.

## 4. How the Data Will Be Used
The dataset I collected from the ClinicalTrials.gov API has 600 completed or terminated clinical trials, with information such as enrollment size, number of study sites, trial duration, phase, sponsor type, study type, and final status. For my first question, I will use **regression** to predict a trial's enrollment based on its number of sites, duration, phase, and sponsor type. If the model predicts enrollment well, it could help planners with budgeting and recruiting. For my second question, I will use **classification** to predict whether a trial is completed or terminated early, using enrollment, number of sites, phase, sponsor type, and study type as features. I will leave out duration because, for terminated trials, it shows when the trial stopped and could give away the answer. Looking at which features are most important can also show which trial characteristics are linked to early termination. Together, these models can help explain trial size and identify trials that may be at higher risk of ending early.

4) (25%) Record a video walking us through your proposed project, the data, and why the questions are of interest to you.

## Video Walkthrough

**Link to my project proposal video** in case it doesn't show up in gradescope: [https://drive.google.com/file/d/16O3OTZKRTSQ1SWPsr1kRxzzaK6T_Xtzx/view?usp=sharing]

[https://app.screencastify.com/watch/Cd0Nj1NpxxahLnTSCf5N]

In [7]:
from IPython.display import HTML

HTML("""
<iframe src="https://drive.google.com/file/d/16O3OTZKRTSQ1SWPsr1kRxzzaK6T_Xtzx/preview" width="640" height="480"></iframe>
""")

## Why This Matters to Me

I'm a bioengineering major, and clinical trials are the step between a promising idea in a lab and a treatment that can actually help people. I find it frustrating that so many trials stop early, because it means the time, money, and effort of researchers and patients can go to waste. I chose cancer trials because of my past co-op where I worked in the lab developing pharmaceuticals for this. If I can find patterns in which trials struggle, even a simple model might point to ways to plan trials better, and that's the kind of problem I'd like to work on in my career.